In [2]:
import pandas as pd


ModuleNotFoundError: No module named 'pandas'

In [ ]:
data1 = pd.read_csv("train.csv")
data2 = pd.read_csv("test.csv")

In [ ]:
data1.head()



In [ ]:
data2.head()

In [ ]:
print(data1.describe())


In [ ]:
print(data2.describe())

In [ ]:
print(data1.columns)
print("*****************")
print(data2.columns)

In [ ]:
print(data1.dtypes)
print("************************")
print(data2.dtypes)

In [ ]:
data1.columns = data1.columns.str.strip()
data2.columns = data2.columns.str.strip()
print(data1.columns.tolist())
print(data2.columns.tolist())

In [ ]:
# Remove useless columns but keep PassengerId
useless_cols = ['VRDeck', 'Spa', 'ShoppingMall', 'FoodCourt', 'RoomService', 'Name']

# Create cleaned datasets
data1_cleaned = data1.drop(columns=useless_cols)
data2_cleaned = data2.drop(columns=useless_cols)

# Check the columns
print("Columns in data1_cleaned:", data1_cleaned.columns.tolist())
print("Columns in data2_cleaned:", data2_cleaned.columns.tolist())


In [ ]:
# For data1
print("Data1 info:")
print(data1.dtypes)
print("\nMissing values in data1:")
print(data1.isnull().sum())



# For data2
print("Data2 info:")
print(data2.dtypes)
print("\nMissing values in data2:")
print(data2.isnull().sum())


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

cat_cols = ['HomePlanet', 'Destination', 'Cabin']
for col in cat_cols:
    if col in data1_cleaned.columns:
        mode = data1_cleaned[col].mode(dropna=True)[0]
        data1_cleaned[col].fillna(mode, inplace=True)
        data2_cleaned[col].fillna(mode, inplace=True)

bool_cols = ['CryoSleep', 'VIP']
for col in bool_cols:
    if col in data1_cleaned.columns:
        data1_cleaned[col] = data1_cleaned[col].map({True:1, False:0, 'True':1, 'False':0})
        data2_cleaned[col] = data2_cleaned[col].map({True:1, False:0, 'True':1, 'False':0})
        mode = data1_cleaned[col].mode()[0]
        data1_cleaned[col].fillna(mode, inplace=True)
        data2_cleaned[col].fillna(mode, inplace=True)

num_cols = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
num_cols = [col for col in num_cols if col in data1_cleaned.columns]
for col in num_cols:
    median = data1_cleaned[col].median()
    data1_cleaned[col].fillna(median, inplace=True)
    data2_cleaned[col].fillna(median, inplace=True)

def split_cabin(df):
    if 'Cabin' in df.columns:
        df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
        df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')

split_cabin(data1_cleaned)
split_cabin(data2_cleaned)

median_cabin = data1_cleaned['CabinNum'].median() if 'CabinNum' in data1_cleaned.columns else 0
if 'CabinNum' in data1_cleaned.columns:
    data1_cleaned['CabinNum'].fillna(median_cabin, inplace=True)
    data2_cleaned['CabinNum'].fillna(median_cabin, inplace=True)

spend_cols = [col for col in ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck'] if col in data1_cleaned.columns]
for df in [data1_cleaned, data2_cleaned]:
    if spend_cols:
        df['TotalSpend'] = df[spend_cols].sum(axis=1)
        df['NoSpend'] = (df['TotalSpend'] == 0).astype(int)
    else:
        df['TotalSpend'] = 0
        df['NoSpend'] = 1

for df in [data1_cleaned, data2_cleaned]:
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')

cat_features = ['HomePlanet', 'Destination', 'Deck', 'Side']
cat_features = [col for col in cat_features if col in data1_cleaned.columns]
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([data1_cleaned[col], data2_cleaned[col]]).astype(str)
    le.fit(combined)
    data1_cleaned[col] = le.transform(data1_cleaned[col].astype(str))
    data2_cleaned[col] = le.transform(data2_cleaned[col].astype(str))

features = [
    'HomePlanet','CryoSleep','Destination','Age','VIP',
    'RoomService','FoodCourt','ShoppingMall','Spa','VRDeck',
    'Deck','Side','CabinNum','TotalSpend','NoSpend','GroupSize'
]
features = [col for col in features if col in data1_cleaned.columns]

X_train = data1_cleaned[features]
y_train = data1_cleaned['Transported']
X_test = data2_cleaned[features]

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf_model.fit(X_tr, y_tr)
val_preds = rf_model.predict(X_val)
print("Validation Accuracy:", accuracy_score(y_val, val_preds))

rf_model.fit(X_train, y_train)
test_preds = rf_model.predict(X_test)

submission = data2_cleaned[['PassengerId']].copy()
submission['Transported'] = test_preds.astype(bool)

submission.to_csv("submission.csv", index=False)
print("Submission file created")
